In [15]:
!pip install faiss-cpu openai


Defaulting to user installation because normal site-packages is not writeable
  Using cached anyio-4.8.0-py3-none-any.whl.metadata (4.6 kB)
  Using cached distro-1.9.0-py3-none-any.whl.metadata (6.8 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached jiter-0.9.0-cp312-cp312-win_amd64.whl.metadata (5.3 kB)
  Using cached pydantic-2.10.6-py3-none-any.whl.metadata (30 kB)
  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
  Using cached tqdm-4.67.1-py3-none-any.whl.metadata (57 kB)
  Using cached httpcore-1.0.7-py3-none-any.whl.metadata (21 kB)
  Using cached h11-0.14.0-py3-none-any.whl.metadata (8.2 kB)
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached pydantic_core-2.27.2-cp312-cp312-win_amd64.whl.metadata (6.7 kB)
   ---------------------------------------- 0.0/13.7 MB ? eta -:--:--
   ----------- ---------------------------- 3.9/13.7 MB 19.5 MB/s eta 0:00:01
   ----------------------------- ---------- 10.0/


[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: C:\Users\r2com\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [ ]:
import json
import numpy as np
import faiss
import os
from openai import OpenAI

# 모든 JSON 데이터 로드
json_files = [f for f in os.listdir("C:/Users/r2com/Documents/GitHub/BeBaek/ChefBot/data/recipes") if f.endswith(".json")]

all_recipes = []
for file in json_files:
    with open(f"C:/Users/r2com/Documents/GitHub/BeBaek/ChefBot/data/recipes/{file}", "r", encoding="utf-8") as f:
        data = json.load(f)
        all_recipes.extend(data)

print(f"총 {len(all_recipes)} 개의 레시피 데이터 로드 완료")

# OpenAI API 키 설정
API_KEY = '' # 깃헙 반영을 위해 공백 처리  
client = OpenAI(api_key=API_KEY)

# 벡터화할 텍스트 데이터 준비
texts = [f"{recipe['name']} {recipe['ingredients']} {' '.join(recipe['recipe'])}" for recipe in all_recipes]

# OpenAI Embedding 생성 (Batch 처리)
def get_embeddings(texts, batch_size=100):
    embeddings = []

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        try:
            response = client.embeddings.create(
                input=batch, 
                model="text-embedding-ada-002"
            )
            batch_embeddings = [item.embedding for item in response.data] 
            embeddings.extend(batch_embeddings)
        except Exception as e:
            print(f"OpenAI API 호출 오류 발생: {e}")

    return np.array(embeddings)

# 모든 레시피 데이터 임베딩 생성
embeddings = get_embeddings(texts)

# FAISS 벡터 DB 구축
d = embeddings.shape[1]  # 벡터 차원 수
index = faiss.IndexFlatL2(d)  # L2 거리 기반 벡터 검색 인덱스
index.add(embeddings)  # 벡터 추가

# FAISS 인덱스 저장
faiss.write_index(index, "./faiss_index/recipes_faiss.index")

print("벡터화 완료 & FAISS 저장 완료")

총 1052 개의 레시피 데이터 로드 완료
벡터화 완료 & FAISS 저장 완료


In [2]:
import faiss

# 저장된 FAISS 인덱스 로드
index_path = "./faiss_index/recipes_faiss.index"
index = faiss.read_index(index_path)

# 벡터 개수 확인
print(f"FAISS 인덱스 로드 완료, 저장된 벡터 개수: {index.ntotal}")

FAISS 인덱스 로드 완료, 저장된 벡터 개수: 1052


In [3]:
import numpy as np
import faiss

# 저장된 FAISS 인덱스 로드
index = faiss.read_index(index_path)

# 테스트용 랜덤 벡터 생성 (벡터 차원 수 맞추기)
d = index.d  # 벡터 차원 수
test_vector = np.random.random((1, d)).astype('float32')

# 가장 가까운 3개 벡터 찾기
D, I = index.search(test_vector, 3)

print(f"검색된 벡터의 인덱스: {I[0]}")
print(f"검색된 벡터의 거리: {D[0]}")


검색된 벡터의 인덱스: [ 99 251 300]
검색된 벡터의 거리: [492.0063  492.19693 492.2034 ]


In [4]:
# 검색된 레시피 확인
search_results = [all_recipes[i] for i in I[0]]

print("검색된 레시피:")
for i, recipe in enumerate(search_results):
    print(f"{i+1}. {recipe['name']} - 재료: {recipe['ingredients']} - 레시피: {recipe['recipe']}")

검색된 레시피:
1. 그릭 요거트 볼 - 재료: 우유 1000ml, 액티비아 1개 - 레시피: ['1. 1, 전기밥솥에 우유와 액티비아를 넣어요', '2. 2, 나무 주걱으로 잘 저어요', '3. 3, 전기밥솥에 보온으로  1시간 정도 두었다가  보온을 끄고  그 상태로 8~9시간 두어요', '4. 4, 8시간후 요거트 완성', '5. 저는 플레인 요거트를  맛보고 싶어서  조금 덜어 두었습니다 (냉장보관)', '6. 아침에  요거트에 딸기잼을  넣고 맛을 보았어요', '7. 딸기 요플레  완전  신선하고 맛있어요', '8. 5, ④의 요거트를 면보에 넣고 묶어요', '9. 6, 이 상태에서 그대로  냉장고에 넣고 (8~9시간 정도 둬요)', '10. 님아 제발 유청은 버리지 마시오!', '11. 요거트 아래로 걸러진  유청은 단백질이라 마셔야죠~', '12. 라씨 만들기  우유 : 유청 = 1 : 1  꿀 1TS 넣기  유산균의 단맛과 향이 느껴져요', '13. 7, 유청이 빠지면 돌로 눌러 놓는다', '14. 8, 그릭 요거트 완성', '15. 꾸덕한 그릭 요거트', '16. 샐러드위에 한스쿱 퍼서 올린다', '17. 레몬즙 살짝 뿌린  새싹에  그릭 요거트  한스쿱 올리기', '18. 단호박 구이  과일 채소와  곁들여 먹으니 웰빙', '19. 한마디로 식감 천재 입니다']
2. 마늘빵 - 재료: 버터 50g (3T), 올리브유 10g (1T), 설탕 10g (1T), 마늘 1T, 파슬리 조금 - 레시피: ['1. 버터를 전자레인지에 20초만 녹여주세요.', '2. 전자레인지에 20초정도 돌린 버터에 올리브유를 넣고 섞어주세요.', '3. 마늘을 넣고 섞는다.', '4. 설탕도 넣고 섞는다.', '5. 식빵을 잘라서 준비한 후', '6. 마늘양념을 앞뒤로 발라주세요. 바른 후 파슬리가루 솔솔. 오븐(180℃)에서 15분간 굽기.', '7. 맛있는 초간단 마늘빵 완성. 식빵혼자 식탁에 덩그러니 놓여있을때는 손이 잘 안갔는데 마늘빵으로 만들어서 책상위에 올려